<a href="https://colab.research.google.com/github/lamvu211/SharedDrive-to-MyDrive/blob/main/Copy_Folder_Google_Drive_to_Google_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Copy Folder Google Drive to Google Drive - 1TouchPro

In [ ]:
#@title Input
from ipywidgets import widgets

dest_text = widgets.Text(description="Your drive", placeholder='Nhập đường link folder Google Drive của bạn')
source_text = widgets.Text(description="Shared drive", placeholder='Nhập đường link folder Google Drive shared')
from_page_text = widgets.Text(description="Từ trang", value="0")
to_page_text = widgets.Text(description="Đến trang", value="0")
max_download_size_text = widgets.Text(description="Tổng dung lượng tối đa(GB)", value="700")
exclude_str_text = widgets.Text(description="Bỏ file, folder có chứa nội dung", value="")

display(dest_text)
display(source_text)
display(from_page_text)
display(to_page_text)
display(max_download_size_text)
display(exclude_str_text)

In [ ]:
#@title Run
import os
import time
import re
import random
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import auth

def execute_with_retry(request, max_retries=5):
    for attempt in range(max_retries):
        try:
            return request.execute()
        except HttpError as err:
            status_code = err.resp.status if hasattr(err, 'resp') else None
            err_str = str(err)
            is_rate_limit = status_code in [403, 429, 500, 503] or 'rateLimitExceeded' in err_str or 'userRateLimitExceeded' in err_str
            if is_rate_limit and attempt < max_retries - 1:
                sleep_time = (2 ** attempt) + random.uniform(0.5, 1.5)
                print(f"[Rate Limit / Busy] Thử lại lần {attempt + 1}/{max_retries} sau {sleep_time:0.1f}s...")
                time.sleep(sleep_time)
                continue
            raise err
        except Exception as e:
            if attempt < max_retries - 1:
                sleep_time = (2 ** attempt) + random.uniform(0.5, 1.5)
                time.sleep(sleep_time)
                continue
            raise e

class DownloadFromDrive:
    def __init__(self):
        self._total_size = 0
        self._limit_size = 0
        self._is_stopped = False
        self.excluded_strings = []

    def get_user_credential(self):
        auth.authenticate_user()
        drive_service = build('drive', 'v3')
        return drive_service

    def extract_folder_id_from_url(self, url):
        if not url:
            return None
        url = url.strip()
        match = re.search(r'/folders/([-\w]{25,})', url)
        if match:
            return match.group(1)
        match = re.search(r'[?&]id=([-\w]{25,})', url)
        if match:
            return match.group(1)
        match = re.search(r'[-\w]{25,}', url)
        if match:
            return match.group(0)
        return None

    def check_if_exists(self, drive_service, dest_folder_id, name):
        try:
            processed_name = name.replace('\\\\', '\\\\\\\\').replace("'", "\\\\'")
            req = drive_service.files().list(
                q=f"'{dest_folder_id}' in parents and name = '{processed_name}' and trashed = false",
                fields='files(id, name)',
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            )
            results = execute_with_retry(req)
            if 'files' in results and len(results['files']) > 0:
                return results['files'][0]['id']
        except Exception as e:
            print(f"Lỗi kiểm tra file tồn tại [{name}]: {e}")
        return ""

    def create_folder(self, drive_service, dest_folder_id, sub_folder_name):
        exist_folder_id = self.check_if_exists(drive_service, dest_folder_id, sub_folder_name)
        if exist_folder_id:
            return exist_folder_id
        
        sub_folder_inf = {
            'name': sub_folder_name,
            'mimeType': 'application/vnd.google-apps.folder',
            'parents': [dest_folder_id]
        }
        try:
            req = drive_service.files().create(
                body=sub_folder_inf,
                fields='id',
                supportsAllDrives=True
            )
            folder = execute_with_retry(req)
            return folder['id']
        except Exception as e:
            print(f"Lỗi tạo thư mục [{sub_folder_name}]: {e}")
            return ""

    def get_childs_from_folder(self, drive_service, folder_id, from_page, to_page):
        files = []
        page_token = None
        query = f"'{folder_id}' in parents and trashed = false"
        if self.excluded_strings and len(self.excluded_strings) > 0:
            not_contains_query = " and ".join([f"not name contains '{ext}'" for ext in self.excluded_strings])
            query += f" and ({not_contains_query})"

        pages = 0
        while True:
            try:
                pages += 1
                req = drive_service.files().list(
                    q=query,
                    orderBy='name, createdTime',
                    fields='files(id, name, mimeType, size), nextPageToken',
                    pageToken=page_token,
                    supportsAllDrives=True,
                    includeItemsFromAllDrives=True
                )
                response = execute_with_retry(req)
                if (from_page < pages <= to_page) or to_page == 0:
                    files.extend(response.get('files', []))

                page_token = response.get('nextPageToken', None)
                if page_token is None or (pages >= to_page > 0):
                    break
            except Exception as e:
                print(f"Lỗi khi đọc danh sách file trang {pages}: {e}")
                break

        print(f"Tổng số mục tìm thấy: {len(files)}")
        return files

    def copy_file(self, drive_service, dest_folder_id, source_file):
        if self._is_stopped:
            return

        if source_file['mimeType'] != 'application/vnd.google-apps.folder':
            body_file_inf = {'parents': [dest_folder_id]}
            if not self.check_if_exists(drive_service, dest_folder_id, source_file['name']):
                try:
                    start_time = time.time()
                    req_copy = drive_service.files().copy(
                        body=body_file_inf,
                        fileId=source_file['id'],
                        supportsAllDrives=True
                    )
                    execute_with_retry(req_copy)
                    end_time = time.time()

                    file_size = int(source_file.get('size', 0))
                    size_mb = file_size / (1024 * 1024)
                    self._total_size += size_mb
                    duration = max(end_time - start_time, 0.001)
                    speed_mb = size_mb / duration
                    print(f"[{source_file['name']}] đã copy. Dung lượng {size_mb:0.2f} MB. Tốc độ {speed_mb:0.2f} MB/s")

                    if self._limit_size > 0 and self._total_size >= (self._limit_size * 1024):
                        print(f"\\n[DỪNG] Đã đạt giới hạn dung lượng {self._limit_size} GB. Dừng quá trình copy an toàn.")
                        self._is_stopped = True
                        return
                except Exception as e:
                    print(f"Lỗi khi copy [{source_file['name']}]: {e}")
            else:
                print(f"[{source_file['name']}] đã tồn tại, bỏ qua.")
        else:
            source_files = self.get_childs_from_folder(drive_service, source_file['id'], 0, 0)
            if source_files and len(source_files) > 0:
                print(f"--- Bắt đầu thư mục: {source_file['name']} ---")
                sub_folder_id = self.create_folder(drive_service, dest_folder_id, source_file['name'])
                if sub_folder_id:
                    self.copy_multiple_files(drive_service, sub_folder_id, source_files)
                print(f"--- Hoàn thành thư mục: {source_file['name']} ---")

    def copy_multiple_files(self, drive_service, dest_folder_id, source_files):
        for source_file in source_files:
            if self._is_stopped:
                break
            self.copy_file(drive_service, dest_folder_id, source_file)

    def copy_drive_to_drive(self, dest_drive_link, source_drive_link, from_page, to_page):
        service = self.get_user_credential()
        start_time = time.time()

        dest_folder_id = self.extract_folder_id_from_url(dest_drive_link)
        source_folder_id = self.extract_folder_id_from_url(source_drive_link)

        if not dest_folder_id:
            print("Lỗi: Không tìm thấy Folder ID trong link đích (Your drive).")
            return
        if not source_folder_id:
            print("Lỗi: Không tìm thấy Folder ID trong link nguồn (Shared drive).")
            return

        try:
            req_folder = service.files().get(fileId=source_folder_id, supportsAllDrives=True)
            source_folder = execute_with_retry(req_folder)
        except Exception as e:
            print(f"Lỗi truy cập thư mục nguồn [{source_folder_id}]: {e}")
            return

        new_dest_folder_id = self.create_folder(service, dest_folder_id, source_folder['name'])
        if not new_dest_folder_id:
            print("Không thể tạo hoặc mở thư mục đích.")
            return

        source_files = self.get_childs_from_folder(service, source_folder_id, from_page, to_page)
        self.copy_multiple_files(service, new_dest_folder_id, source_files)
        end_time = time.time()

        size_gb = self._total_size / 1024
        duration = max(int(end_time - start_time), 1)
        speed_mb = self._total_size / duration
        print(f"\\n=== Hoàn tất. Tổng dung lượng: {size_gb:0.2f} GB. Tổng thời gian: {duration}s. Tốc độ trung bình: {speed_mb:0.2f} MB/s ===")

# Main
dest_drive_link = dest_text.value
source_drive_link = source_text.value
try:
    from_page = int(from_page_text.value)
except ValueError:
    from_page = 0
try:
    to_page = int(to_page_text.value)
except ValueError:
    to_page = 0

downloader = DownloadFromDrive()
try:
    downloader._limit_size = float(max_download_size_text.value)
except ValueError:
    downloader._limit_size = 700.0

downloader.excluded_strings = [ext.strip() for ext in exclude_str_text.value.split(",") if ext.strip()]
downloader.copy_drive_to_drive(dest_drive_link, source_drive_link, from_page, to_page)
